In [30]:
%pip install pydicom
%pip install SimpleITK
%pip install git+https://github.com/Radiomics/pyradiomics.git
%pip install idc-index
%pip install pynrrd
%pip install dcmqi

  Cloning https://github.com/Radiomics/pyradiomics.git to /tmp/pip-req-build-1equc91q
  Running command git clone --filter=blob:none --quiet https://github.com/Radiomics/pyradiomics.git /tmp/pip-req-build-1equc91q
  Resolved https://github.com/Radiomics/pyradiomics.git to commit 8ed579383b44806651c463d5e691f3b2b57522ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 3.7 MB/s eta 0:00:00


In [32]:
import time
import pandas as pd
import pydicom
from idc_index import IDCClient
import os
import csv
import SimpleITK as sitk
import subprocess
import shutil
from google.colab import auth, files
from google.cloud import bigquery
import matplotlib.pyplot as plt
import nrrd
import numpy as np
from radiomics import featureextractor

In [33]:
BASE_DIR = "/content"
DOWNLOAD_DIR = os.path.join(BASE_DIR, "nlst")
COHORT_CSV = os.path.join(BASE_DIR, "patient_cohort.csv")
HEADERS_CSV = os.path.join(BASE_DIR, "ct_headers.csv")
FEATURES_CSV = os.path.join(BASE_DIR, "radiomic_features.csv")
TIMING_LOG = os.path.join(BASE_DIR, "timing_log.csv")

In [34]:
my_ProjectID = "nlst-radiomics"
os.environ["GCP_PROJECT_ID"] = my_ProjectID

auth.authenticate_user()
client = bigquery.Client(my_ProjectID)

lung_nodule_seg_series_descriptions = [
    'AIMI lung and nodule radiologist 5 corrected segmentation',
    'AIMI lung and nodule radiologist 4 corrected segmentation',
    'AIMI lung and nodule radiologist 8 corrected segmentation',
    'AIMI lung and nodule AI segmentation'
]

query = f"""
SELECT
    DISTINCT seg.SeriesInstanceUID as SEG_SeriesInstanceUID,
    seg.SeriesDescription AS SEG_SeriesDescription,
    ct.SeriesInstanceUID as CT_SeriesInstanceUID,
    ct.StudyInstanceUID as StudyInstanceUID,
    ct.PatientID
FROM
    `bigquery-public-data.idc_v23.dicom_all` AS ct
JOIN
    `bigquery-public-data.idc_v23.dicom_all` AS seg
ON
    ct.PatientID = seg.PatientID
WHERE
    ct.collection_id = 'nlst'
    AND seg.collection_id = 'nlst'
    AND ct.Modality = 'CT'
    AND seg.Modality = 'SEG'
    AND seg.SeriesDescription IN ({', '.join([f"'{s}'" for s in lung_nodule_seg_series_descriptions])})
    AND EXISTS (
        SELECT 1
        FROM UNNEST(seg.ReferencedSeriesSequence) as ref_seq
        WHERE ct.SeriesInstanceUID = ref_seq.SeriesInstanceUID
    )
"""

df = client.query(query).to_dataframe()

# Currently there are more SEG series than CT series; this is because some CT series have AI segmentation and radiologist corrected segmentation
# "Keep" the radiologist corrected segmentation by creating a priority column
ignore_desc = "AIMI lung and nodule AI segmentation"
df['Priority'] = (df['SEG_SeriesDescription'] == ignore_desc).astype(int)
df_sorted = df.sort_values(by=['CT_SeriesInstanceUID', 'Priority'])
df_clean = df_sorted.drop_duplicates(subset='CT_SeriesInstanceUID', keep='first')

final_df = df_clean[['PatientID', 'StudyInstanceUID', 'CT_SeriesInstanceUID', 'SEG_SeriesInstanceUID']]
final_df = final_df.sort_values(by=['PatientID', 'StudyInstanceUID'])
final_df.to_csv(COHORT_CSV, index=False)
final_df['NumNodules'] = 0
final_df['Status'] = "Unprocessed"
final_df.to_csv(COHORT_CSV, index=False)
print(f"PatientIDs and SeriesInstanceUIDs saved to {COHORT_CSV}")

PatientIDs and SeriesInstanceUIDs saved to /content/patient_cohort.csv


In [35]:
def extract_ct_header_info(pid, study_uid, ct_uid):
    base_headers = [
        "Manufacturer", "ManufacturerModelName", "SliceThickness", "KVP",
        "DataCollectionDiameter", "FilterType", "FocalSpots", "ConvolutionKernel",
        "ExposureTime", "XRayTubeCurrent", "Exposure", "PixelSpacing"
    ]
    try:
        ct_series_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid, "CT_" + ct_uid)
        dicom_files = [f for f in os.listdir(ct_series_dir) if f.endswith('.dcm')]
        ct_file_path = os.path.join(ct_series_dir, dicom_files[0])
        ct_img = pydicom.dcmread(ct_file_path, stop_before_pixels=True)
        ct_info = {}
        for header in base_headers:
            ct_info[header] = ct_img.get(header, "N/A")
    except Exception as e:
        print(f"Failed to read CT header information for patient {pid}: {study_uid}: {e}")
        raise e

    csv_headers = ["PatientID", "StudyInstanceUID"] + base_headers
    file_exists = os.path.exists(HEADERS_CSV)
    with open(HEADERS_CSV, "a", newline='') as f:
        w = csv.DictWriter(f, fieldnames=csv_headers)
        if not file_exists:
            w.writeheader()
        row = {
            "PatientID": pid,
            "StudyInstanceUID": study_uid
        }
        row.update(ct_info)
        w.writerow(row)

In [36]:
def convert_CT_to_nrrd(pid, ct_uid, study_uid):
    try:
        reader = sitk.ImageSeriesReader()
        ct_series_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid, "CT_" + ct_uid)
        dicom_names = reader.GetGDCMSeriesFileNames(ct_series_dir)
        reader.SetFileNames(dicom_names)
        ct_image = reader.Execute()
        output_path = os.path.join(DOWNLOAD_DIR, pid, study_uid, "CT.nrrd")
        sitk.WriteImage(ct_image, output_path)
    except Exception as e:
        print(f"Failed to convert CT to .nrrd for patient {pid}: {study_uid}: {e}")
        raise e

def convert_SEG_to_nrrd(pid, seg_uid, study_uid):
    seg_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid, "SEG_" + seg_uid)
    seg_files = [f for f in os.listdir(seg_dir) if f.endswith('.dcm')]
    if not seg_files:
        raise FileNotFoundError(f"SEG DICOM not found in {seg_dir}")
    input_dicom = os.path.join(seg_dir, seg_files[0])
    output_dir = os.path.join(DOWNLOAD_DIR, pid, study_uid)
    command = [
        "segimage2itkimage",
        "--inputDICOM", input_dicom,
        "--outputDirectory", output_dir,
        "--outputType", "nrrd",
        "--prefix", "SEG"
    ]
    try:
        subprocess.run(command, check=True, capture_output=True, text=True)
        os.rename(os.path.join(output_dir, "SEG-1.nrrd"), os.path.join(output_dir, "lung.nrrd"))
        os.rename(os.path.join(output_dir, "SEG-2.nrrd"), os.path.join(output_dir, "nodules.nrrd"))
    except subprocess.CalledProcessError as e:
        print(f"SEG conversion failed for patient {pid}, study {study_uid}: {e.stderr}")
        raise e
    except FileNotFoundError as e:
        print(f"Renaming the .nrrd files failed for patient {pid}, study {study_uid}: {e}")
        raise e

def separate_nodules(pid, study_uid):
    nodules_path = os.path.join(DOWNLOAD_DIR, pid, study_uid, "nodules.nrrd")
    nodules_image = sitk.ReadImage(nodules_path)

    cc_filter = sitk.ConnectedComponentImageFilter()
    labeled_image = cc_filter.Execute(nodules_image > 0)
    num_nodules = cc_filter.GetObjectCount()
    for i in range(1, num_nodules + 1):
        nodule_mask = sitk.BinaryThreshold(labeled_image, lowerThreshold=i, upperThreshold=i, insideValue=1, outsideValue=0)
        output_path = os.path.join(DOWNLOAD_DIR, pid, study_uid, f"nodule_{i}.nrrd")
        sitk.WriteImage(nodule_mask, output_path)
    return num_nodules

def delete_series(pid, study_uid, ct_uid, seg_uid):
    study_path = os.path.join(DOWNLOAD_DIR, pid, study_uid)
    meta_json = os.path.join(study_path, "SEG-meta.json")
    ct_path = os.path.join(study_path, "CT.nrrd")
    lung_path = os.path.join(study_path, "lung.nrrd")
    nodules_path = os.path.join(study_path, "nodules.nrrd")
    lung_aligned_path = os.path.join(study_path, "lung_aligned.nrrd")
    nodules_aligned_path = os.path.join(study_path, "nodules_aligned.nrrd")
    for path in [meta_json, ct_path, lung_path, nodules_path, lung_aligned_path, nodules_aligned_path]:
        if os.path.exists(path):
            os.remove(path)
    ct_dir = os.path.join(study_path, "CT_" + ct_uid)
    seg_dir = os.path.join(study_path, "SEG_" + seg_uid)
    for folder in [ct_dir, seg_dir]:
        if os.path.exists(folder):
            shutil.rmtree(folder)

In [37]:
from __future__ import annotations

from pydicom.config import settings

from radiomics.featureextractor import RadiomicsFeatureExtractor

import collections
import logging
import threading
from itertools import chain

from radiomics import (
    generalinfo,
    getFeatureClasses,
    getImageTypes,
    getParameterValidationFiles,
    imageoperations,
)

logger = logging.getLogger(__name__)

class _SingletonGeometryTolerance:
    _instance = None
    _initialized = False
    _lock = threading.Lock()

    def __new__(cls, *_args, **_kwargs):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, tolerance=None):
        if not self._initialized:
            with self._lock:
                if not self._initialized:
                    self.geometryTolerance = tolerance
                    _SingletonGeometryTolerance._initialized = True

class SaveCropFeatureExtractor(RadiomicsFeatureExtractor):
    def execute(
        self,
        imageFilepath,
        maskFilepath,
        label=None,
        label_channel=None,
        voxelBased=False,
    ):
        """
        Compute radiomics signature for provide image and mask combination. It comprises of the following steps:

        1. Image and mask are loaded and normalized/resampled if necessary.
        2. Validity of ROI is checked using :py:func:`~imageoperations.checkMask`, which also computes and returns the
           bounding box.
        3. If enabled, provenance information is calculated and stored as part of the result. (Not available in voxel-based
           extraction)
        4. Shape features are calculated on a cropped (no padding) version of the original image. (Not available in
           voxel-based extraction)
        5. If enabled, resegment the mask based upon the range specified in ``resegmentRange`` (default None: resegmentation
           disabled).
        6. Other enabled feature classes are calculated using all specified image types in ``_enabledImageTypes``. Images
           are cropped to tumor mask (no padding) after application of any filter and before being passed to the feature
           class.
        7. The calculated features is returned as ``collections.OrderedDict``.

        :param imageFilepath: SimpleITK Image, or string pointing to image file location
        :param maskFilepath: SimpleITK Image, or string pointing to labelmap file location
        :param label: Integer, value of the label for which to extract features. If not specified, last specified label
            is used. Default label is 1.
        :param label_channel: Integer, index of the channel to use when maskFilepath yields a SimpleITK.Image with a vector
            pixel type. Default index is 0.
        :param voxelBased: Boolean, default False. If set to true, a voxel-based extraction is performed, segment-based
            otherwise.
        :returns: dictionary containing calculated signature ("<imageType>_<featureClass>_<featureName>":value).
            In case of segment-based extraction, value type for features is float, if voxel-based, type is SimpleITK.Image.
            Type of diagnostic features differs, but can always be represented as a string.
        """
        _settings = self.settings.copy()

        tolerance = _settings.get("geometryTolerance")
        additionalInfo = _settings.get("additionalInfo", False)
        resegmentShape = _settings.get("resegmentShape", False)

        if label is not None:
            _settings["label"] = label
        else:
            label = _settings.get("label", 1)

        if label_channel is not None:
            _settings["label_channel"] = label_channel

        if _SingletonGeometryTolerance().geometryTolerance != tolerance:
            self._setTolerance()

        if additionalInfo:
            generalInfo = generalinfo.GeneralInfo()
            generalInfo.addGeneralSettings(_settings)
            generalInfo.addEnabledImageTypes(self.enabledImagetypes)
        else:
            generalInfo = None

        if voxelBased:
            _settings["voxelBased"] = True
            kernelRadius = _settings.get("kernelRadius", 1)
            logger.info("Starting voxel based extraction")
        else:
            kernelRadius = 0

        logger.info("Calculating features with label: %d", label)
        logger.debug("Enabled images types: %s", self.enabledImagetypes)
        logger.debug("Enabled features: %s", self.enabledFeatures)
        logger.debug("Current settings: %s", _settings)

        # 1. Load the image and mask
        featureVector = collections.OrderedDict()
        image, mask = self.loadImage(
            imageFilepath, maskFilepath, generalInfo, **_settings
        )

        # 2. Check whether loaded mask contains a valid ROI for feature extraction and get bounding box
        # Raises a ValueError if the ROI is invalid
        boundingBox, correctedMask = imageoperations.checkMask(image, mask, **_settings)

        # Update the mask if it had to be resampled
        if correctedMask is not None:
            if generalInfo is not None:
                generalInfo.addMaskElements(image, correctedMask, label, "corrected")
            mask = correctedMask

        logger.debug("Image and Mask loaded and valid, starting extraction")

        # 5. Resegment the mask if enabled (parameter regsegmentMask is not None)
        resegmentedMask = None
        if _settings.get("resegmentRange", None) is not None:
            resegmentedMask = imageoperations.resegmentMask(image, mask, **_settings)

            # Recheck to see if the mask is still valid, raises a ValueError if not
            boundingBox, correctedMask = imageoperations.checkMask(
                image, resegmentedMask, **_settings
            )

            if generalInfo is not None:
                generalInfo.addMaskElements(
                    image, resegmentedMask, label, "resegmented"
                )

        # 3. Add the additional information if enabled
        if generalInfo is not None:
            featureVector.update(generalInfo.getGeneralInfo())

        # if resegmentShape is True and resegmentation has been enabled, update the mask here to also use the
        # resegmented mask for shape calculation (e.g. PET resegmentation)
        if resegmentShape and resegmentedMask is not None:
            mask = resegmentedMask

        if not voxelBased:
            # 4. If shape descriptors should be calculated, handle it separately here
            featureVector.update(
                self.computeShape(image, mask, boundingBox, **_settings)
            )

        # (Default) Only use resegemented mask for feature classes other than shape
        # can be overridden by specifying `resegmentShape` = True
        if not resegmentShape and resegmentedMask is not None:
            mask = resegmentedMask

        # 6. Calculate other enabled feature classes using enabled image types
        # Make generators for all enabled image types
        logger.debug("Creating image type iterator")
        imageGenerators = []
        for imageType, customKwargs in self.enabledImagetypes.items():
            args = _settings.copy()
            args.update(customKwargs)
            msg = f'Adding image type "{imageType}" with custom settings: {customKwargs!s}'
            logger.info(msg)
            imageGenerators = chain(
                imageGenerators,
                getattr(imageoperations, f"get{imageType}Image")(image, mask, **args),
            )

        logger.debug("Extracting features")
        # Calculate features for all (filtered) images in the generator
        for originputImage, imageTypeName, inputKwargs in imageGenerators:
            logger.info("Calculating features for %s image", imageTypeName)
            inputImage, inputMask = imageoperations.cropToTumorMask(
                originputImage, mask, boundingBox, padDistance=kernelRadius
            )

            if isinstance(maskFilepath, str):
                sitk.WriteImage(sitk.Cast(inputMask, sitk.sitkUInt8), maskFilepath[:-5] + "_cropped.nrrd", useCompression=True)
                print("Saving cropped nodule .nrrd file")

            featureVector.update(
                self.computeFeatures(
                    inputImage, inputMask, imageTypeName, **inputKwargs
                )
            )

        logger.debug("Features extracted")

        return featureVector

In [38]:
def extract_features(pid, study_id, num_nodules):
    ct_path = os.path.join(DOWNLOAD_DIR, pid, study_id, "CT.nrrd")
    for i in range(1, num_nodules + 1):
        nodule_path = os.path.join(DOWNLOAD_DIR, pid, study_id, f"nodule_{i}.nrrd")
        settings = {
            'resegmentRange': [-1000, 2000], # Excludes air and bone (HU scale) for feature calculation
            'label': 2, # Label 2 is nodule
            'interpolator': 'sitkBSpline',
            'resampledPixelSpacing': [1.0, 1.0, 1.0]
        }

        try:
            extractor = SaveCropFeatureExtractor(**settings)
            features = extractor.execute(ct_path, nodule_path)

            row_data = {"PatientID": pid, "StudyInstanceUID": study_id, "NoduleID": i}
            row_data.update(features)

            file_exists = os.path.exists(FEATURES_CSV)
            with open(FEATURES_CSV, "a", newline='') as f:
                w = csv.DictWriter(f, fieldnames=row_data.keys())
                if not file_exists:
                    w.writeheader()
                w.writerow(row_data)
        except Exception as e:
            print(f"Failed to extract features for patient {pid}, study {study_id}, nodule {i}: {e}")
            raise e

In [39]:
def align_masks(pid, study_uid):
    timestep_path = os.path.join(DOWNLOAD_DIR, pid, study_uid)
    ct = sitk.ReadImage(os.path.join(timestep_path, "CT.nrrd"))
    lung_mask = sitk.ReadImage(os.path.join(timestep_path, "lung.nrrd"))
    nodule_mask = sitk.ReadImage(os.path.join(timestep_path, "nodules.nrrd"))
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(ct)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    aligned_lung_mask = resampler.Execute(lung_mask)
    aligned_nodule_mask = resampler.Execute(nodule_mask)
    sitk.WriteImage(aligned_lung_mask, os.path.join(timestep_path, "lung_aligned.nrrd"))
    sitk.WriteImage(aligned_nodule_mask, os.path.join(timestep_path, "nodules_aligned.nrrd"))

def plot_ct_with_masks(pid, study_uid):
    timestep_path = os.path.join(DOWNLOAD_DIR, pid, study_uid)
    ct, _ = nrrd.read(os.path.join(timestep_path, f"CT.nrrd"))
    lung, _ = nrrd.read(os.path.join(timestep_path, f"lung.nrrd"))
    nodules, _ = nrrd.read(os.path.join(timestep_path, f"nodules.nrrd"))

    slice_counts = np.sum(nodules > 0, axis=(0,1))
    z_slice = np.argmax(slice_counts)
    ct_slice = ct[:, :, z_slice]
    lung_slice = lung[:, :, z_slice]
    nodule_slice = nodules[:, :, z_slice]

    fig, axes = plt.subplots(2,2,figsize=(12,12))
    axes[0,0].imshow(ct_slice, cmap='gray', vmin=-1000, vmax=400, origin='lower')
    axes[0,0].set_title("Raw CT Slice")
    axes[0,0].axis('off')

    axes[0,1].imshow(ct_slice, cmap='gray', vmin=-1000, vmax=400, origin='lower')
    lung_mask = np.ma.masked_where(lung_slice == 0, lung_slice)
    axes[0,1].imshow(lung_mask, cmap='Blues', alpha=0.3, origin='lower', vmin=0, vmax=1)
    nodule_mask = np.ma.masked_where(nodule_slice == 0, nodule_slice)
    axes[0,1].imshow(nodule_mask, cmap='Reds', alpha=0.6, origin='lower', vmin=0, vmax=1)
    axes[0,1].set_title("CT Slice with Lung and Nodule Masks")
    axes[0,1].axis('off')

    axes[1,0].imshow(nodule_slice, cmap='gray', origin='lower')
    axes[1,0].set_title("Nodule Mask Only")
    axes[1,0].axis('off')

    nodule_voxels = ct_slice[nodule_slice > 0]
    axes[1,1].hist(nodule_voxels, bins=30, color='crimson', edgecolor='black')
    axes[1,1].set_title("Nodule HU Histogram")
    axes[1,1].set_xlabel("Hounsfield Units (HU)")
    axes[1,1].set_ylabel("Voxel Count")

    fig.suptitle(f"Patient ID: {pid}, Study UID: {study_uid}")
    plt.tight_layout()
    plt.savefig(os.path.join(timestep_path, f"CT_with_masks.png"), dpi=300, bbox_inches='tight')
    plt.close(fig)

In [40]:
client = IDCClient()

def log_performance(pid, study_uid, t_download, t_header, t_convert, t_features, t_total):
    file_exists = os.path.exists(TIMING_LOG)
    with open(TIMING_LOG, "a", newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["PatientID", "StudyInstanceUID", "Download_Sec", "Header_Sec", "Convert_Sec", "Features_Sec", "Total_Sec"])
        writer.writerow([pid, study_uid, round(t_download, 2), round(t_header, 2), round(t_convert, 2), round(t_features, 2), round(t_total, 2)])

# batch_size is number of series to be processed
def process_series(batch_size=2, plot=False):
    df = pd.read_csv(COHORT_CSV)
    work_queue = df[df['Status'] == 'Unprocessed'].head(batch_size)
    total_in_batch = len(work_queue)
    if total_in_batch == 0:
        print("No unprocessed data found.")
        return

    # Use enumerate to get the current count (i) starting at 1
    for i, (index, row) in enumerate(work_queue.iterrows(), 1):
        pid = str(row['PatientID'])
        study_uid = str(row['StudyInstanceUID'])
        ct_uid = str(row['CT_SeriesInstanceUID'])
        seg_uid = str(row['SEG_SeriesInstanceUID'])
        print(f"\n[{i}/{total_in_batch}] Processing Patient: {pid} , Study: {study_uid}")
        print("-" * 30)
        start_total = time.time()

        try:
            print(f"Downloading CT and SEG series")
            start_time = time.time()
            client.download_from_selection(seriesInstanceUID=ct_uid, downloadDir=BASE_DIR)
            client.download_from_selection(seriesInstanceUID=seg_uid, downloadDir=BASE_DIR)
            t_download = time.time() - start_time

            print(f"Extracting CT header information")
            start_time = time.time()
            extract_ct_header_info(pid, study_uid, ct_uid)
            t_header = time.time() - start_time

            print(f"Converting CT and SEG to .nrrd format")
            start_time = time.time()
            convert_CT_to_nrrd(pid, ct_uid, study_uid)
            convert_SEG_to_nrrd(pid, seg_uid, study_uid)
            num_nodules = separate_nodules(pid, study_uid)
            df.at[index, 'NumNodules'] = num_nodules
            t_convert = time.time() - start_time

            if plot:
                print(f"Aligning masks and plotting CT with masks")
                align_masks(pid, study_uid)
                plot_ct_with_masks(pid, study_uid)

            print(f"Extracting radiomic features")
            start_time = time.time()
            extract_features(pid, study_uid, num_nodules)
            t_features = time.time() - start_time
            print(f"Deleting series data")
            delete_series(pid, study_uid, ct_uid, seg_uid)
            t_total = time.time() - start_total

            df.at[index, 'Status'] = 'Completed'
            print(f"DONE. Times: Download: {t_download:.1f}s | "
                  f"Header: {t_header:.1f}s | "
                  f"Convert: {t_convert:.1f}s | "
                  f"Features: {t_features:.1f}s | "
                  f"Total: {t_total:.1f}s")
            log_performance(pid, study_uid, t_download, t_header, t_convert, t_features, t_total)

        except Exception as e:
            print(f"Failed to process {pid}, Study: {study_uid}: {e}")
            df.at[index, 'Status'] = f"Failed: {str(e)[:50]}"

        df.to_csv(COHORT_CSV, index=False)

    print(f"\nProcessed {batch_size} series.")

In [41]:
process_series(batch_size=3, plot=True)


[1/3] Processing Patient: 100012 , Study: 1.2.840.113654.2.55.238034941445508011386463276954045956831
------------------------------


Extracting CT header information
Converting CT and SEG to .nrrd format
Aligning masks and plotting CT with masks
Failed to process 100012, Study: 1.2.840.113654.2.55.238034941445508011386463276954045956831: 'bool' object is not callable

[2/3] Processing Patient: 100012 , Study: 1.2.840.113654.2.55.3832109283939010833855886500020760484
------------------------------


Extracting CT header information
Converting CT and SEG to .nrrd format
Aligning masks and plotting CT with masks
Failed to process 100012, Study: 1.2.840.113654.2.55.3832109283939010833855886500020760484: 'bool' object is not callable

[3/3] Processing Patient: 100147 , Study: 1.2.840.113654.2.55.133032926508606330165457723341736723804
------------------------------


Extracting CT header information
Converting CT and SEG to .nrrd format
Aligning masks and plotting CT with masks
Failed to process 100147, Study: 1.2.840.113654.2.55.133032926508606330165457723341736723804: 'bool' object is not callable

Processed 3 series.
